# AuditMayorista — Auditoría de E-Commerce Mayorista · Posadas, Misiones

**Tesis:** *Factores estratégicos asociados al desarrollo del comercio electrónico en tiendas mayoristas de bienes de consumo masivo de Posadas, Misiones*  
**Autor:** Diego Enrique Arce · **Director:** Dr. Carlos Roberto Brys  
**Institución:** Universidad Nacional de Misiones — FCE — Maestría en AEN

---
**Uso:** Menú → **Entorno de ejecución → Ejecutar todo** `Ctrl+F9`


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diarce/Auditoria/blob/main/AuditMayorista_Colab.ipynb)

> **Ejecución directa:** haga clic en el badge o abra la URL:  
> `https://colab.research.google.com/github/diarce/Auditoria/blob/main/AuditMayorista_Colab.ipynb`

El notebook clona automáticamente el repositorio [`diarce/Auditoria`](https://github.com/diarce/Auditoria) y ejecuta el sistema completo sin configuración adicional.

## ① Configuración

In [ ]:
import warnings, os, subprocess, sys

# Suprimir avisos de compatibilidad de paquetes del sistema
warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow.*',
                        category=DeprecationWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning,
                        module='jupyter_client')
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Clonar o actualizar repositorio
REPO = 'https://github.com/diarce/Tesis_Maen.git'
DIR  = '/content/Tesis_Maen'

if os.path.isdir(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '--quiet'], check=True)
    print('Repositorio actualizado')
else:
    subprocess.run(['git', 'clone', '--quiet', REPO, DIR], check=True)
    print('Repositorio clonado')

os.chdir(DIR)
sys.path.insert(0, DIR)

# Instalar dependencias
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
print(f'Python {sys.version.split()[0]} | Entorno listo')


## ② Universo del estudio

Carga el relevamiento de empresas desde `data/universo_mayoristas_posadas.csv` — fuente canónica del estudio. Clasifica cada empresa según su nivel de madurez digital y define el tratamiento metodológico apropiado.

In [ ]:
from modules.universo import (cargar_universo, empresas_auditables,
                              empresas_excluidas, tabla_universo_html)
from modules.storage  import DatabaseManager
from modules.universo import registrar_en_db
from config           import DB_PATH
from IPython.display  import display, HTML
import pandas as pd

universo = cargar_universo()
db       = DatabaseManager(DB_PATH)

n_audit  = len(empresas_auditables(universo))
n_excl   = len(empresas_excluidas(universo))
n_campo  = sum(1 for e in universo if 'PENDIENTE' in e.estado_auditoria)
n_verif  = sum(1 for e in universo if e.estado_auditoria == 'VERIFICAR')

print(f'Universo relevado : {len(universo)} empresas')
print(f'  Auditables (QA) : {n_audit}  — tienen plataforma web propia')
print(f'  Excluidas del QA: {n_excl}  — solo redes sociales (documentar como brecha digital)')
print(f'  Pendientes campo: {n_campo}  — sin URL verificada aun')
print(f'  Por verificar   : {n_verif}  — datos incompletos')

display(HTML(tabla_universo_html(universo)))


## ③ Empresas excluidas del QA — justificación metodológica

Las empresas sin plataforma de e-commerce propia son excluidas de la auditoría QA pero **se documentan** porque su ausencia de comercio digital es evidencia empírica directa de la brecha que estudia la tesis.

In [ ]:
excluidas = empresas_excluidas(universo)

filas = []
for e in excluidas:
    filas.append({
        'ID'        : e.id,
        'Empresa'   : e.nombre,
        'Ciudad'    : e.ciudad,
        'Presencia' : e.tipo_presencia,
        'Plataforma': e.plataforma_cms,
        'Justificacion': e.notas_eticas[:120],
    })

df_excl = pd.DataFrame(filas)
display(df_excl.style
    .set_caption('Empresas excluidas del instrumento QA — incluir en analisis cualitativo')
    .set_table_styles([{
        'selector': 'th',
        'props': [('background','#1a1a1a'),('color','white'),
                  ('padding','5px 8px'),('font-size','11px')]
    },{
        'selector': 'td',
        'props': [('font-size','11px'),('padding','4px 8px')]
    }])
)
print()
print('NOTA METODOLOGICA: La presencia exclusiva en redes sociales')
print('es evidencia de nivel D en la escala de madurez digital.')
print('Incluir en la seccion de resultados como hallazgo empirico,')
print('no como limitacion del estudio.')


## ④ Auditoría QA — empresas con plataforma web

Las tres empresas con e-commerce propio (Makro, DIA%, Vital) tienen `robots.txt` restrictivo, lo que impide el acceso automatizado. Se aplica el **protocolo de auditoría demo** que replica el proceso de compra mayorista con datos representativos del mercado. Para cargar datos de una auditoría manual real, usar la Sección ⑤.

In [ ]:
from modules.demo import run_demo, _SIMULATED_SITES
from modules.universo import registrar_en_db

# Registrar las empresas auditables en la BD
n = registrar_en_db(universo, db)
print(f'{n} empresas auditables registradas en la BD')

# Ejecutar auditoría demo
print('Ejecutando auditoría demo...')
for s in _SIMULATED_SITES:
    db.clear_site_results(s['id'])
run_demo(db)

resultados = db.get_audit_results()
sitios     = db.get_sites()
print(f'Completada: {len(sitios)} sitios | {len(resultados)} indicadores evaluados')

# Nota sobre el protocolo manual
print()
print('NOTA: Para cargar resultados de una auditoria manual,')
print('completar la plantilla CSV y ejecutar la Seccion V.')


## ⑤ Matriz de resultados QA

In [ ]:
from modules.reporter import build_anon_map
from config import QA_DIMENSIONS

scores     = db.get_dimension_scores()
sitios_bd  = db.get_sites()
resultados = db.get_audit_results()

ids_con_res = {r['site_id'] for r in resultados}
sitios_act  = [s for s in sitios_bd if s['id'] in ids_con_res]
site_map    = build_anon_map(sitios_act)
pesos       = {d: QA_DIMENSIONS[d]['weight'] for d in QA_DIMENSIONS}

by_site = {}
for r in scores:
    by_site.setdefault(r['site_id'], {})[r['dimension_id']] = r

def calc_icc(dd):
    ws = sum(dd[d]['avg_compliance']*pesos[d] for d in pesos if d in dd)
    wp = sum(pesos[d] for d in pesos if d in dd)
    return round(ws/wp, 2) if wp else 0

dim_ids = sorted(QA_DIMENSIONS.keys())
filas = []
for sid, dd in sorted(by_site.items()):
    row = {'Plataforma': site_map.get(sid, sid)}
    for d in dim_ids:
        row[d] = round(dd.get(d, {}).get('avg_compliance', 0), 2)
    row['ICC'] = calc_icc(dd)
    filas.append(row)

df = pd.DataFrame(filas).set_index('Plataforma')

def _color(v):
    if   v >= 2.5: return 'background:#1a4a6e;color:white;font-weight:bold'
    elif v >= 1.5: return 'background:#5a8fb5;color:white;font-weight:bold'
    elif v >  0.0: return 'background:#c8d8e8;color:#333'
    else:          return 'background:#f0f0f0;color:#aaa'

display(df.style.applymap(_color).format('{:.2f}')
    .set_caption('Matriz QA | Escala 0-3 | ICC = Indice de Calidad Compuesto')
    .set_table_styles([{
        'selector':'th',
        'props':[('background','#1a1a1a'),('color','white'),
                 ('font-family','Georgia,serif'),('padding','6px 10px')]
    }])
)
display(HTML(
    '<div style="font-size:11px;margin:4px 0">'
    '<span style="background:#1a4a6e;color:white;padding:2px 8px;margin-right:4px">Pleno &ge;2,5</span>'
    '<span style="background:#5a8fb5;color:white;padding:2px 8px;margin-right:4px">Parcial 1,5&#8211;2,4</span>'
    '<span style="background:#c8d8e8;color:#333;padding:2px 8px;margin-right:4px">No cumple &lt;1,5</span>'
    '<span style="background:#f0f0f0;color:#aaa;padding:2px 8px">N/A (0)</span>'
    '</div>'
))


## ⑥ Importar auditoría manual (opcional)

Cuando el investigador complete la auditoría manual de un sitio, subir el CSV completado aquí. **Formato:** usar `plantilla_auditoria_manual.csv` del repositorio.

In [ ]:
# Descomente para importar una auditoria manual

# from google.colab import files
# from modules.importer import AuditImporter
#
# uploaded = files.upload()          # seleccionar el CSV completado
# csv_path = list(uploaded.keys())[0]
#
# importer = AuditImporter(db, log_fn=print)
# resultado = importer.from_csv(csv_path)
#
# print(f"Importadas : {resultado['insertadas']} filas")
# print(f"Sitios nuevos: {resultado['sitios_nuevos']}")
# if resultado['errores']:
#     for e in resultado['errores'][:5]:
#         print(f'  ERROR: {e}')

print('Seccion lista. Descomente el bloque para importar datos de campo.')


## ⑦ Visualizaciones académicas

In [ ]:
from modules.reporter import HTMLReporter

rep = HTMLReporter(db)

# ── Barras ICC ────────────────────────────────────────────────────────────────
svg_b = rep._seccion_barras_svg(sitios_act, by_site)
display(HTML(
    '<div style="font-family:Georgia,serif">'
    '<div style="font-size:14px;font-weight:bold;margin:10px 0 4px">'
    'Indice de Calidad Compuesto (ICC) por plataforma</div>'
    + svg_b + '</div>'
))


In [ ]:
# ── Radar multi-sitio ────────────────────────────────────────────────────────
svg_r = rep._seccion_radar_svg(sitios_act, by_site)
display(HTML(
    '<div style="font-family:Georgia,serif">'
    '<div style="font-size:14px;font-weight:bold;margin:10px 0 4px">'
    'Perfil de cumplimiento por dimension — Radar</div>'
    + svg_r + '</div>'
))


In [ ]:
# ── Mapa de calor ────────────────────────────────────────────────────────────
res_act = [r for r in resultados if r['site_id'] in {s['id'] for s in sitios_act}]
html_hm = rep._seccion_heatmap_html(sitios_act, res_act)
display(HTML(
    '<div style="font-family:Georgia,serif">'
    '<div style="font-size:14px;font-weight:bold;margin:10px 0 4px">'
    'Mapa de calor — Indicadores x Plataformas</div>'
    + html_hm + '</div>'
))


## ⑧ Exportar informe HTML

In [ ]:
import base64

path_inf  = HTMLReporter(db).generate()
contenido = path_inf.read_text(encoding='utf-8')
b64       = base64.b64encode(contenido.encode('utf-8')).decode()
kb        = path_inf.stat().st_size // 1024
href      = f'data:text/html;charset=utf-8;base64,{b64}'

display(HTML(
    f'<div style="font-family:Georgia,serif;border:1px solid #ccc;'
    f'border-radius:6px;padding:16px;max-width:480px">'
    f'<div style="font-size:14px;font-weight:bold;margin-bottom:8px">'
    f'Informe HTML generado</div>'
    f'<div style="font-size:12px;color:#555;margin-bottom:12px">'
    f'{path_inf.name} ({kb} KB)</div>'
    f'<a href="{href}" download="{path_inf.name}" '
    f'style="background:#1a1a1a;color:white;padding:10px 20px;'
    f'border-radius:4px;text-decoration:none;font-size:13px">'
    f'Descargar informe HTML</a></div>'
))

# Opcional: guardar en Google Drive
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# shutil.copy(path_inf, f'/content/drive/MyDrive/{path_inf.name}')


## Cita académica

```bibtex
@software{AuditMayorista2026,
  author  = {Arce, Diego Enrique},
  title   = {{AuditMayorista}: Herramienta de auditoria automatizada
             de calidad funcional para plataformas de comercio
             electronico mayorista},
  year    = {2026},
  version = {5.0},
  url     = {https://github.com/diarce/Auditoria}
}
```
> Boton **'Cite this repository'** disponible en el panel derecho de GitHub (generado desde `CITATION.cff`).
